<a href="https://colab.research.google.com/github/rekhaannapurna/Paddy-Disease-Detection/blob/main/Colab/SVM_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from pathlib import Path

features_path = Path(
    "/content/drive/MyDrive/Paddy_Disease_Project/features/_MobileNetV2"
)

X_train = np.load(features_path / "X_train.npy")
y_train = np.load(features_path / "y_train.npy")
X_valid = np.load(features_path / "X_valid.npy")
y_valid = np.load(features_path / "y_valid.npy")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)



X_train: (8326, 1280)
y_train: (8326,)
X_valid: (2081, 1280)
y_valid: (2081,)


In [4]:
from fastai.vision.all import *

learn = load_learner(
    "/content/drive/MyDrive/Paddy_Disease_Project/MobileNetV2_paddy_baseline.pkl"
)

print(learn.model)
print(learn.dls.vocab)

/usr/local/lib/python3.13/dist-packages/fastai/learner.py:456: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [5]:


from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import time

classes = [
    'bacterial_leaf_blight',
    'bacterial_leaf_streak',
    'bacterial_panicle_blight',
    'blast',
    'brown_spot',
    'dead_heart',
    'downy_mildew',
    'hispa',
    'normal',
    'tungro'
]

svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,
    random_state=42
)

start_time = time.time()
svm_model.fit(X_train, y_train)
svm_train_time = time.time() - start_time

start_time = time.time()
svm_preds = svm_model.predict(X_valid)
svm_inference_time = time.time() - start_time

svm_accuracy = accuracy_score(y_valid, svm_preds)

print(f"SVM Training time: {svm_train_time:.4f} seconds")
print(f"SVM Inference time: {svm_inference_time:.4f} seconds")
print(f"\nSVM Accuracy: {svm_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_valid,
    svm_preds,
    target_names=classes,
    digits=4
))

SVM Training time: 13.4814 seconds
SVM Inference time: 1.0402 seconds

SVM Accuracy: 97.55%

Classification Report:
                          precision    recall  f1-score   support

   bacterial_leaf_blight     0.9800    0.9703    0.9751       101
   bacterial_leaf_streak     0.9851    1.0000    0.9925        66
bacterial_panicle_blight     0.9828    1.0000    0.9913        57
                   blast     0.9803    0.9694    0.9749       360
              brown_spot     0.9810    0.9718    0.9764       213
              dead_heart     0.9966    0.9897    0.9931       292
            downy_mildew     0.9138    0.9381    0.9258       113
                   hispa     0.9757    0.9698    0.9727       331
                  normal     0.9672    0.9878    0.9774       328
                  tungro     0.9726    0.9682    0.9704       220

                accuracy                         0.9755      2081
               macro avg     0.9735    0.9765    0.9750      2081
            weighted avg

In [6]:
import joblib

model_path = "/content/drive/MyDrive/Paddy_Disease_Project/models"
!mkdir -p "$model_path"

joblib.dump(
    svm_model,
    f"{model_path}/model_2_svm.pkl"
)

print("SVM model saved successfully.")

SVM model saved successfully.
